# PyTorch Transformer-Only Practice Workbook

[Open in Colab](https://colab.research.google.com/github/mudit1729/mlwiki-colabs/blob/main/pytorch-code-samples/pytorch-transformer-implementations.ipynb)

Transformer-only implementation drills for scaled dot-product attention, multi-head attention, positional encoding, transformer encoder/decoder blocks, Vision Transformers, DETR-style object detection, and transformer segmentation heads.

Source wiki page: [transformer implementation notes](https://auto-driving-wiki.up.railway.app/page/wiki/syntheses/pytorch-code-samples/paper-implementations-transformers)

RoPE and modern components (KV-cache, GQA, sliding-window, FlashAttention, MoE, RMSNorm, TinyGPT) are in the [Transformer Part 2 notebook](https://colab.research.google.com/github/mudit1729/mlwiki-colabs/blob/main/pytorch-code-samples/pytorch-transformer-implementations-2.ipynb).

This workbook is **PyTorch-only** and intentionally does **not** include completed answers. It gives you boilerplate signatures plus test helpers. Fill the TODOs first, then uncomment each `run_exercise_XX_tests()` call.

Each test helper validates your implementation against a **trusted oracle** — a PyTorch built-in (e.g. `F.scaled_dot_product_attention`) where one exists, or an independent inline reference and differential properties (causality, batch independence, grad flow, reduction to a known case) where none does — instead of hard-coded magic numbers.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__)
print("device", device)


def assert_shape(x, shape):
    assert tuple(x.shape) == tuple(shape), f"expected shape {shape}, got {tuple(x.shape)}"


def assert_finite_tensor(x):
    assert torch.isfinite(x).all(), "tensor contains non-finite values"


def assert_close(a, b, atol=1e-5, rtol=1e-5):
    torch.testing.assert_close(a, b, atol=atol, rtol=rtol)


# Workbook setup smoke test.
_x = torch.randn(2, 3, device=device)
assert_shape(_x, (2, 3))
assert_finite_tensor(_x)


## Practice Flow

1. Read the exercise prompt and shape hints.
2. Fill the TODO implementation body.
3. Uncomment `run_exercise_XX_tests()`.
4. Run the cell and fix the implementation until the tests pass.
5. Explain tensor shapes out loud before writing each attention reshape.

The test helpers compare your output to a trusted oracle (PyTorch built-in or an independent inline reference) plus differential properties, so a passing run is strong evidence of correctness rather than a shape-only smoke test. Keep this workbook answer-free while practicing.


## Exercise 01: Scaled Dot-Product Attention

Implement attention from first principles.

Expected shapes:

- `q`: `(batch, heads, query_len, head_dim)`
- `k`: `(batch, heads, key_len, head_dim)`
- `v`: `(batch, heads, key_len, head_dim)`
- `mask`: broadcastable to `(batch, heads, query_len, key_len)`; `True` means keep and `False` means block
- output: `(batch, heads, query_len, head_dim)`

Concepts tested: scale by `sqrt(head_dim)`, mask before softmax, causal masking, and attention probabilities.


In [ ]:
# Exercise 01: Scaled Dot-Product Attention
# Fill the TODOs, then run the test helper at the bottom of this cell.

def scaled_dot_product_attention(q, k, v, mask=None, dropout_p=0.0, is_causal=False):
    """Return (output, attention_weights).

    q, k, v are shaped (B, H, Tq/Tk, Dh). mask uses True=keep, False=block.
    """
    # TODO: compute scores = q @ k.transpose(-2, -1) / sqrt(Dh).
    # TODO: apply optional causal mask for query/key lengths.
    # TODO: apply optional boolean mask by filling blocked logits with -inf.
    # TODO: softmax over key dimension, optional dropout, then weighted sum over v.
    raise NotImplementedError("Implement scaled_dot_product_attention")


def run_exercise_01_tests():
    """Run after filling the TODO implementation above.

    Oracle: torch's own F.scaled_dot_product_attention (bool attn_mask uses
    True=keep, the same convention as this notebook). We assert our output
    matches the built-in for plain, causal, and padding-mask cases, then add
    differential properties the built-in can't check (weights, grad flow).
    """
    torch.manual_seed(0)
    q = torch.randn(2, 3, 4, 8, device=device)
    k = torch.randn(2, 3, 5, 8, device=device)
    v = torch.randn(2, 3, 5, 8, device=device)

    # shape + finite + weights normalize over keys
    out, attn = scaled_dot_product_attention(q, k, v)
    assert_shape(out, (2, 3, 4, 8))
    assert_shape(attn, (2, 3, 4, 5))
    assert_finite_tensor(out)
    assert_close(attn.sum(dim=-1), torch.ones(2, 3, 4, device=device))

    # ORACLE: plain attention == torch built-in
    ref = F.scaled_dot_product_attention(q, k, v)
    assert_close(out, ref)

    # ORACLE: causal attention == built-in is_causal (Tq == Tk so the masks align)
    qc = torch.randn(2, 3, 5, 8, device=device)
    out_c, attn_c = scaled_dot_product_attention(qc, k, v, is_causal=True)
    ref_c = F.scaled_dot_product_attention(qc, k, v, is_causal=True)
    assert_close(out_c, ref_c)
    # causal weights are strictly upper-triangular zero (token t can't see t+1)
    upper = torch.triu(torch.ones(5, 5, dtype=torch.bool, device=device), diagonal=1)
    assert torch.all(attn_c[..., upper] == 0)

    # ORACLE: padding-mask case == built-in with a bool attn_mask (True=keep)
    mask = torch.ones(2, 1, 4, 5, dtype=torch.bool, device=device)
    mask[..., -1] = False                                   # block the last key for every query
    out_m, attn_m = scaled_dot_product_attention(q, k, v, mask=mask)
    ref_m = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    assert_close(out_m, ref_m)
    # blocked keys get exactly zero weight; remaining weights still sum to 1
    assert_close(attn_m[..., -1], torch.zeros_like(attn_m[..., -1]))
    assert_close(attn_m.sum(dim=-1), torch.ones(2, 3, 4, device=device))

    # grad flow: gradients reach q, k, v and are finite
    qg = q.clone().requires_grad_(True)
    kg = k.clone().requires_grad_(True)
    vg = v.clone().requires_grad_(True)
    og, _ = scaled_dot_product_attention(qg, kg, vg)
    og.sum().backward()
    for g in (qg.grad, kg.grad, vg.grad):
        assert g is not None
        assert_finite_tensor(g)
    print("exercise 01 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_01_tests()


## Exercise 02: Multi-Head Attention

Build a reusable multi-head attention module on top of Exercise 01.

Expected shapes:

- input `x`: `(batch, seq_len, embed_dim)` for self-attention
- optional `context`: `(batch, context_len, embed_dim)` for cross-attention
- output: `(batch, seq_len, embed_dim)`

Concepts tested: Q/K/V projections, head splitting, transpose/contiguous/view discipline, output projection, and optional cross-attention.


In [ ]:
# Exercise 02: Multi-Head Attention
# Fill the TODOs, then run the test helper at the bottom of this cell.

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must divide num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        # TODO: define q_proj, k_proj, v_proj, out_proj, and dropout.

    def _split_heads(self, x):
        """Convert (B, T, D) to (B, H, T, Dh)."""
        # TODO: reshape and transpose.
        raise NotImplementedError("Implement _split_heads")

    def _merge_heads(self, x):
        """Convert (B, H, T, Dh) to (B, T, D)."""
        # TODO: transpose, make contiguous if needed, then reshape.
        raise NotImplementedError("Implement _merge_heads")

    def forward(self, x, context=None, mask=None, is_causal=False):
        """Self-attention when context=None; cross-attention otherwise."""
        # TODO: project Q from x; project K/V from context if provided else x.
        # TODO: split heads, call scaled_dot_product_attention, merge heads, apply out_proj.
        raise NotImplementedError("Implement forward")


def run_exercise_02_tests():
    """Run after filling the TODO implementation above.

    No weight-compatible PyTorch built-in exists (nn.MultiheadAttention packs
    Q/K/V into one matrix). Oracle = recompute the layer using ITS OWN
    projection weights via F.scaled_dot_product_attention; the module's output
    must match. Run in .eval() so dropout is off. Then add differential
    properties: cross-attention, causality, batch independence, grad flow.
    """
    torch.manual_seed(0)
    mha = MultiHeadAttention(embed_dim=32, num_heads=4).to(device).eval()
    H, Dh = mha.num_heads, mha.head_dim

    def split(t):
        B, T, _ = t.shape
        return t.reshape(B, T, H, Dh).transpose(1, 2)

    def merge(t):
        B, h, T, dh = t.shape
        return t.transpose(1, 2).reshape(B, T, h * dh)

    def oracle(x, context=None, mask=None, is_causal=False):
        kv = x if context is None else context
        q = split(mha.q_proj(x))
        k = split(mha.k_proj(kv))
        v = split(mha.v_proj(kv))
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal)
        return mha.out_proj(merge(o))

    x = torch.randn(2, 6, 32, device=device)
    memory = torch.randn(2, 9, 32, device=device)

    # self-attention matches the oracle
    out = mha(x)
    assert_shape(out, (2, 6, 32))
    assert_finite_tensor(out)
    assert_close(out, oracle(x))

    # cross-attention (context != None) matches the oracle
    cross = mha(x, context=memory)
    assert_shape(cross, (2, 6, 32))
    assert_close(cross, oracle(x, context=memory))

    # causal self-attention matches the oracle (Tq == Tk)
    assert_close(mha(x, is_causal=True), oracle(x, is_causal=True))

    # batch independence: perturbing batch item 0 leaves item 1 unchanged
    out_full = mha(x)
    x2 = x.clone()
    x2[0] += 10.0 * torch.randn_like(x2[0])
    assert_close(mha(x2)[1], out_full[1])

    # grad flow
    xg = x.clone().requires_grad_(True)
    mha(xg).sum().backward()
    assert xg.grad is not None
    assert_finite_tensor(xg.grad)
    print("exercise 02 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_02_tests()


## Exercise 03: Sinusoidal Positional Encoding

Implement the original transformer sinusoidal positional encoding table, then wrap it in a `PositionalEncoding` `nn.Module`.

Expected shapes:

- `sinusoidal_positional_encoding(max_len, d_model)` -> `(max_len, d_model)` table.
- `PositionalEncoding(d_model).forward(x)` where `x` is `(batch, seq_len, d_model)` -> same shape, with positions added.

Concepts tested: even/odd dimensions, exponential frequency schedule, broadcasting, device/dtype handling, and registering the table as a non-trainable **buffer** (not a `Parameter`) so it travels with `.to(device)` and `state_dict` but never gets gradients.

This `PositionalEncoding` module is reused in the ViT (Ex06), DETR (Ex07), and segmentation (Ex08) exercises in place of a learned positional embedding.

Reference: [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) and the [PyTorch tutorial](https://docs.pytorch.org/tutorials/beginner/transformer_tutorial.html). Matching their `div_term` is the key; handle odd `d_model` without a shape error.


In [ ]:
# Exercise 03: Sinusoidal Positional Encoding
# Fill the TODOs, then run the test helper at the bottom of this cell.

def sinusoidal_positional_encoding(max_len, d_model, device=None, dtype=torch.float32):
    """Return a (max_len, d_model) sinusoidal positional encoding table."""
    # TODO: create positions shaped (max_len, 1).
    # TODO: create div terms for even dimensions.
    # TODO: fill sin on even indices and cos on odd indices.
    # TODO: handle odd d_model without shape errors.
    raise NotImplementedError("Implement sinusoidal_positional_encoding")


class PositionalEncoding(nn.Module):
    """Wrap the table in a module that adds positions to an input sequence.

    forward(x): x is (B, T, d_model); add the first T rows of the table.
    The table is a registered BUFFER (non-trainable, moves with .to(device),
    saved in state_dict) — NOT an nn.Parameter.
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # TODO: build the (max_len, d_model) table via sinusoidal_positional_encoding.
        # TODO: register it as a buffer named "pe" (shape it (1, max_len, d_model) for broadcasting).
        raise NotImplementedError("Implement PositionalEncoding.__init__")

    def forward(self, x):
        # TODO: return x + the positional slice for the first x.size(1) steps.
        raise NotImplementedError("Implement PositionalEncoding.forward")


def run_exercise_03_tests():
    """Run after filling the TODO implementations above.

    No PyTorch built-in for sinusoidal PE, so the oracle is an independent
    reference computed inline (an explicit double loop over the closed form).
    We assert the table matches it, check the fixed-point properties, then
    validate the new PositionalEncoding module (Task B): correct slice added,
    'pe' registered as a buffer (not a Parameter), and .to(device) support.
    """
    max_len, d_model = 8, 6
    pe = sinusoidal_positional_encoding(max_len, d_model, device=device)
    assert_shape(pe, (max_len, d_model))
    assert_finite_tensor(pe)

    # ORACLE: independent reference via the explicit closed form
    ref = torch.zeros(max_len, d_model, device=device)
    for pos in range(max_len):
        for i in range(d_model):
            angle = pos / (10000.0 ** ((2 * (i // 2)) / d_model))
            ref[pos, i] = math.sin(angle) if i % 2 == 0 else math.cos(angle)
    assert_close(pe, ref)

    # properties: sin/cos are bounded, and position 0 is (0, 1, 0, 1, ...)
    assert torch.all((pe >= -1.0) & (pe <= 1.0))
    assert_close(pe[0, 0::2], torch.zeros(3, device=device))   # sin(0) == 0
    assert_close(pe[0, 1::2], torch.ones(3, device=device))    # cos(0) == 1

    # odd d_model must not crash
    pe_odd = sinusoidal_positional_encoding(4, 5, device=device)
    assert_shape(pe_odd, (4, 5))
    assert_finite_tensor(pe_odd)

    # ---- Task B: the PositionalEncoding nn.Module ----
    pe_mod = PositionalEncoding(d_model, max_len=max_len).to(device)
    # forward(zeros) returns exactly the table slice; matches the function form
    out = pe_mod(torch.zeros(2, max_len, d_model, device=device))
    assert_shape(out, (2, max_len, d_model))
    assert_close(out[0], pe)
    assert_close(pe_mod(torch.zeros(1, 3, d_model, device=device))[0], pe[:3])
    # adds (does not replace): forward(x) == x + table
    xr = torch.randn(2, max_len, d_model, device=device)
    assert_close(pe_mod(xr), xr + pe.unsqueeze(0))

    # the table is a registered BUFFER, not a Parameter
    assert "pe" in pe_mod.state_dict()
    assert len(list(pe_mod.parameters())) == 0
    # .to(device) moved the buffer onto the right device
    assert pe_mod.pe.device.type == device.type
    print("exercise 03 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_03_tests()


## Exercise 04: Transformer Encoder Layer

Implement a pre-norm transformer encoder block.

Expected shape: `(batch, seq_len, embed_dim)` in and out.

Concepts tested: residual connections, layer norm placement, self-attention, feed-forward expansion, dropout, and source masks.


In [ ]:
# Exercise 04: Transformer Encoder Layer
# Fill the TODOs, then run the test helper at the bottom of this cell.

class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.0):
        super().__init__()
        # TODO: define norm1, self_attn, norm2, feed-forward MLP, and dropout.

    def forward(self, x, src_mask=None):
        """Pre-norm encoder: x + self_attn(norm1(x)), then x + mlp(norm2(x))."""
        # TODO: implement residual self-attention block.
        # TODO: implement residual MLP block.
        raise NotImplementedError("Implement TransformerEncoderLayer.forward")


def run_exercise_04_tests():
    """Run after filling the TODO implementation above.

    Oracle: an inline reference that applies the SAME pre-norm formula using
    the layer's own submodules (norm1/attn/norm2/mlp). With dropout off this
    must reproduce forward() exactly. Plus shape, grad flow, and train-vs-eval
    determinism (no dropout -> identical outputs).
    """
    torch.manual_seed(0)
    layer = TransformerEncoderLayer(embed_dim=32, num_heads=4, dropout=0.0).to(device).eval()
    x = torch.randn(2, 6, 32, device=device)

    out = layer(x)
    assert_shape(out, (2, 6, 32))
    assert_finite_tensor(out)

    # ORACLE: inline pre-norm reference using the layer's own submodules
    h = x + layer.attn(layer.norm1(x))
    ref = h + layer.mlp(layer.norm2(h))
    assert_close(out, ref)

    # grad flow
    xg = x.clone().requires_grad_(True)
    layer(xg).sum().backward()
    assert xg.grad is not None
    assert_finite_tensor(xg.grad)

    # with dropout=0, train and eval give identical outputs
    layer.train()
    out_train = layer(x)
    layer.eval()
    assert_close(out_train, layer(x))
    print("exercise 04 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_04_tests()


## Exercise 05: Transformer Decoder Layer

Implement a pre-norm transformer decoder block with causal self-attention and encoder-decoder cross-attention.

Expected shapes:

- `tgt`: `(batch, target_len, embed_dim)`
- `memory`: `(batch, source_len, embed_dim)`
- output: `(batch, target_len, embed_dim)`

Concepts tested: causal masks, self-attention, cross-attention, residual ordering, and MLP block reuse.


In [ ]:
# Exercise 05: Transformer Decoder Layer
# Fill the TODOs, then run the test helper at the bottom of this cell.

def make_causal_mask(seq_len, device=None):
    """Return a boolean mask shaped (1, 1, seq_len, seq_len), True=keep."""
    # TODO: return lower-triangular keep mask.
    raise NotImplementedError("Implement make_causal_mask")


class TransformerDecoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.0):
        super().__init__()
        # TODO: define norm layers, self_attn, cross_attn, MLP, and dropout.

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        """Pre-norm decoder with causal self-attention and cross-attention."""
        # TODO: apply causal self-attention to tgt.
        # TODO: apply cross-attention from tgt to memory.
        # TODO: apply residual MLP.
        raise NotImplementedError("Implement TransformerDecoderLayer.forward")


def run_exercise_05_tests():
    """Run after filling the TODO implementations above.

    Oracle: an inline pre-norm reference using the layer's own submodules
    (causal self-attn -> cross-attn -> MLP) must reproduce forward() with
    dropout off. Then a built-in-free CAUSALITY oracle: perturbing a future
    target token must not change earlier output rows (token t can't see t+1).
    """
    mask = make_causal_mask(5, device=device)
    assert_shape(mask, (1, 1, 5, 5))
    assert mask.dtype == torch.bool
    assert mask[0, 0, 4, 0].item() is True
    assert mask[0, 0, 0, 4].item() is False

    torch.manual_seed(0)
    layer = TransformerDecoderLayer(embed_dim=32, num_heads=4, dropout=0.0).to(device).eval()
    tgt = torch.randn(2, 5, 32, device=device)
    memory = torch.randn(2, 7, 32, device=device)

    out = layer(tgt, memory)
    assert_shape(out, (2, 5, 32))
    assert_finite_tensor(out)

    # ORACLE: inline pre-norm reference (causal self-attn, cross-attn, MLP)
    cm = make_causal_mask(tgt.shape[1], device=device)
    h1 = tgt + layer.self_attn(layer.norm1(tgt), mask=cm)
    h2 = h1 + layer.cross_attn(layer.norm2(h1), context=memory)
    ref = h2 + layer.mlp(layer.norm3(h2))
    assert_close(out, ref)

    # CAUSALITY ORACLE (no built-in needed): perturb the future-most target
    # token; the earlier output rows must be byte-for-byte unchanged.
    base = layer(tgt, memory)
    tgt2 = tgt.clone()
    tgt2[:, 4] += 5.0 * torch.randn_like(tgt2[:, 4])        # change the last target token
    assert_close(layer(tgt2, memory)[:, :4], base[:, :4])   # rows 0..3 unaffected

    # grad flow
    tg = tgt.clone().requires_grad_(True)
    layer(tg, memory).sum().backward()
    assert tg.grad is not None
    assert_finite_tensor(tg.grad)
    print("exercise 05 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_05_tests()


## Exercise 06: Vision Transformer Patch Embedding And Tiny ViT

Implement patch embedding and a minimal ViT classifier using your encoder layer.

Expected shapes:

- image batch: `(batch, channels, height, width)`
- patch tokens: `(batch, num_patches, embed_dim)`
- logits: `(batch, num_classes)`

Concepts tested: patchify with `Conv2d`, class token, positional encoding, encoder stack, and classification head.

**Deviation note:** canonical ViT uses a *learned* `nn.Parameter` positional embedding. Here we deliberately reuse the Exercise 03 sinusoidal `PositionalEncoding` module instead, to exercise it across the vision models.


In [ ]:
# Exercise 06: Vision Transformer Patch Embedding And Tiny ViT
# Fill the TODOs, then run the test helper at the bottom of this cell.

class ViTPatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must divide patch_size"
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        # TODO: define a Conv2d projection with kernel_size=stride=patch_size.

    def forward(self, x):
        """Return patch tokens shaped (B, num_patches, embed_dim)."""
        # TODO: project image to patch grid, flatten spatial dimensions, transpose to tokens.
        raise NotImplementedError("Implement ViTPatchEmbedding.forward")


class TinyViTClassifier(nn.Module):
    def __init__(self, image_size=16, patch_size=4, in_channels=3, embed_dim=32, num_heads=4, depth=2, num_classes=10):
        super().__init__()
        # TODO: define patch_embed, cls_token, encoder layers, norm, and head.
        # TODO: for positions, use the Exercise 03 PositionalEncoding(embed_dim, max_len=num_patches + 1)
        #       (covers the CLS token + all patch tokens). DEVIATION: canonical ViT uses a LEARNED
        #       nn.Parameter pos_embed; here we deliberately use fixed sinusoidal positions instead.

    def forward(self, x):
        # TODO: create patch tokens, prepend class token, add sinusoidal positional encoding.
        # TODO: pass through encoder layers and classify the final class token.
        raise NotImplementedError("Implement TinyViTClassifier.forward")


def run_exercise_06_tests():
    """Run after filling the TODO implementations above.

    Patch-embed oracle: the Conv2d patchify is mathematically identical to an
    nn.Linear applied to unfolded/flattened patches with the conv weight
    reshaped to (D, C*P*P) — assert they match. Then TinyViT: shape (B,classes),
    grad flow, CLS-only readout, and (Task B) that it uses the sinusoidal
    PositionalEncoding module rather than a learned pos_embed Parameter.
    """
    torch.manual_seed(0)
    patch = ViTPatchEmbedding(image_size=16, patch_size=4, in_channels=3, embed_dim=32).to(device)
    images = torch.randn(2, 3, 16, 16, device=device)
    tokens = patch(images)
    assert_shape(tokens, (2, 16, 32))
    assert_finite_tensor(tokens)

    # ORACLE: Conv2d patch embedding == Linear on unfolded patches with the conv weight reshaped
    B, C, Hh, Ww = images.shape
    P = 4
    W = patch.proj.weight                                   # (D, C, P, P)
    bias = patch.proj.bias
    x = images.reshape(B, C, Hh // P, P, Ww // P, P)
    x = x.permute(0, 2, 4, 1, 3, 5).reshape(B, (Hh // P) * (Ww // P), C * P * P)
    ref = x @ W.reshape(W.shape[0], -1).t() + bias          # (B, N, D)
    assert_close(tokens, ref)

    # TinyViT full model: shape, finiteness, grad
    model = TinyViTClassifier(image_size=16, patch_size=4, in_channels=3, embed_dim=32,
                              num_heads=4, depth=2, num_classes=5).to(device).eval()
    logits = model(images)
    assert_shape(logits, (2, 5))
    assert_finite_tensor(logits)
    ig = images.clone().requires_grad_(True)
    model(ig).sum().backward()
    assert ig.grad is not None
    assert_finite_tensor(ig.grad)

    # Task B: uses sinusoidal PositionalEncoding, NOT a learned pos_embed Parameter
    assert isinstance(model.pos_enc, PositionalEncoding)
    assert not any(name == "pos_embed" for name, _ in model.named_parameters())
    assert "pos_enc.pe" in model.state_dict()
    # the PE table is the real sinusoidal table over CLS + N patch tokens
    n_tokens = model.patch_embed.num_patches + 1
    assert_close(model.pos_enc.pe[0], sinusoidal_positional_encoding(n_tokens, 32, device=device))
    print("exercise 06 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_06_tests()


## Exercise 07: DETR-Style Transformer Object Detector

Implement the shape skeleton of a DETR-style detector: image patch tokens are encoded, learned object queries decode against image memory, and heads predict class logits plus normalized boxes.

Expected shapes:

- image batch: `(batch, channels, image_size, image_size)`
- class logits: `(batch, num_queries, num_classes + 1)` where the extra class is no-object/background
- boxes: `(batch, num_queries, 4)` normalized to `[0, 1]` as `(cx, cy, w, h)`

Concepts tested: encoder memory, learned object queries, non-causal decoder self-attention, cross-attention, object classification head, and box regression head.

**Deviation note:** real DETR adds a *2D sine* positional encoding over the feature grid (and keeps the object queries learned). Here we deliberately use the 1D sinusoidal `PositionalEncoding` over the flattened image tokens — a simplification. The object queries remain learned, as in DETR.


In [ ]:
# Exercise 07: DETR-Style Transformer Object Detector
# Fill the TODOs, then run the test helper at the bottom of this cell.

class TinyDETRDetector(nn.Module):
    def __init__(
        self,
        image_size=16,
        patch_size=4,
        in_channels=3,
        embed_dim=32,
        num_heads=4,
        encoder_depth=1,
        decoder_depth=1,
        num_queries=5,
        num_classes=3,
    ):
        super().__init__()
        # TODO: define patch embedding and learned object queries (nn.Parameter).
        # TODO: for image-token positions, use PositionalEncoding(embed_dim, max_len=num_patches).
        #       DEVIATION: real DETR uses a 2D sine PE over the feature grid and keeps the object
        #       queries learned; here we use a 1D sinusoidal PE over the flattened tokens (a
        #       deliberate simplification). The object queries stay learned, as in DETR.
        # TODO: define encoder layers, decoder layers, class head (num_classes + 1), and box head.

    def forward(self, images):
        # TODO: patchify images into memory tokens and add sinusoidal image positions.
        # TODO: run encoder layers over image memory.
        # TODO: expand object queries and run non-causal decoder layers against memory.
        # TODO: return class logits and sigmoid-normalized boxes.
        raise NotImplementedError("Implement TinyDETRDetector.forward")


def run_exercise_07_tests():
    """Run after filling the TODO implementation above.

    No built-in to compare against, so we pin correctness with properties:
    class-logit shape (B, Q, num_classes+1), box shape (B, Q, 4) with every box
    in [0, 1] (sigmoid), and grad flow to the image. Task B also checks that
    image tokens get sinusoidal positions instead of a learned image_pos.
    """
    torch.manual_seed(0)
    model = TinyDETRDetector(
        image_size=16,
        patch_size=4,
        in_channels=3,
        embed_dim=32,
        num_heads=4,
        encoder_depth=1,
        decoder_depth=1,
        num_queries=6,
        num_classes=4,
    ).to(device).eval()
    images = torch.randn(2, 3, 16, 16, device=device)
    class_logits, boxes = model(images)
    assert_shape(class_logits, (2, 6, 5))                   # num_classes + 1 (no-object)
    assert_shape(boxes, (2, 6, 4))
    assert_finite_tensor(class_logits)
    assert_finite_tensor(boxes)
    assert torch.all((boxes >= 0.0) & (boxes <= 1.0))       # sigmoid range

    # grad flow to the input image
    ig = images.clone().requires_grad_(True)
    cl, bx = model(ig)
    (cl.sum() + bx.sum()).backward()
    assert ig.grad is not None
    assert_finite_tensor(ig.grad)

    # Task B: image tokens use sinusoidal (1D) positions, not a learned image_pos Parameter
    assert isinstance(model.image_pos, PositionalEncoding)
    assert not any(name == "image_pos" for name, _ in model.named_parameters())
    assert "image_pos.pe" in model.state_dict()
    n = model.patch_embed.num_patches
    assert_close(model.image_pos.pe[0], sinusoidal_positional_encoding(n, 32, device=device))
    print("exercise 07 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_07_tests()


## Exercise 08: Transformer Segmentation Model

Implement a simple ViT/SegFormer-style segmentation model: patch tokens go through transformer encoder layers, a per-patch classifier predicts class logits, and logits are upsampled to image resolution.

Expected shapes:

- image batch: `(batch, channels, image_size, image_size)`
- segmentation logits: `(batch, num_classes, image_size, image_size)`

Concepts tested: patch token encoder, dense per-token prediction, token-grid reshape, channel-first logits, and upsampling.

**Deviation note:** SegFormer deliberately drops positional encoding (its Mix-FFN conv supplies position). Here we add the Exercise 03 sinusoidal `PositionalEncoding` in place of a learned `pos_embed` — a deliberate deviation to keep the positional treatment consistent across the vision exercises.


In [ ]:
# Exercise 08: Transformer Segmentation Model
# Fill the TODOs, then run the test helper at the bottom of this cell.

class TransformerSegmentationModel(nn.Module):
    def __init__(
        self,
        image_size=16,
        patch_size=4,
        in_channels=3,
        embed_dim=32,
        num_heads=4,
        depth=2,
        num_classes=6,
    ):
        super().__init__()
        # TODO: define patch embedding, encoder layers, norm, and per-patch classifier.
        # TODO: for positions, use PositionalEncoding(embed_dim, max_len=num_patches).
        #       DEVIATION: SegFormer actually drops positional encoding entirely (its
        #       convolutional FFN supplies position); here we deliberately add fixed
        #       sinusoidal positions in place of a learned pos_embed.

    def forward(self, images):
        # TODO: encode image patches (add sinusoidal positions first).
        # TODO: predict per-patch logits.
        # TODO: reshape patch logits to a 2D grid and upsample to image resolution.
        raise NotImplementedError("Implement TransformerSegmentationModel.forward")


def run_exercise_08_tests():
    """Run after filling the TODO implementation above.

    Properties pin correctness: dense output is (B, num_classes, H, W) at full
    image resolution, finite, and gradients flow to the input. Task B checks
    the model uses the sinusoidal PositionalEncoding instead of a learned
    pos_embed Parameter.
    """
    torch.manual_seed(0)
    model = TransformerSegmentationModel(
        image_size=16,
        patch_size=4,
        in_channels=3,
        embed_dim=32,
        num_heads=4,
        depth=2,
        num_classes=6,
    ).to(device).eval()
    images = torch.randn(2, 3, 16, 16, device=device)
    logits = model(images)
    assert_shape(logits, (2, 6, 16, 16))                   # channel-first, full resolution
    assert_finite_tensor(logits)

    # grad flow to the input image
    ig = images.clone().requires_grad_(True)
    model(ig).sum().backward()
    assert ig.grad is not None
    assert_finite_tensor(ig.grad)

    # Task B: sinusoidal PositionalEncoding buffer in place of a learned pos_embed Parameter
    assert isinstance(model.pos_enc, PositionalEncoding)
    assert not any(name == "pos_embed" for name, _ in model.named_parameters())
    assert "pos_enc.pe" in model.state_dict()
    n = model.patch_embed.num_patches
    assert_close(model.pos_enc.pe[0], sinusoidal_positional_encoding(n, 32, device=device))
    print("exercise 08 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_08_tests()


## After You Finish

Use the source wiki page to compare your implementation against the expected architecture details:

- [[wiki/syntheses/pytorch-code-samples/paper-implementations-transformers]]
- [[wiki/syntheses/pytorch-code-samples/hard]]
- [[wiki/syntheses/pytorch-code-samples/cheatsheet]]
- [[wiki/syntheses/pytorch-code-samples/paper-implementations-transformers-2]] (RoPE + modern components)

Follow-up drills after the basics pass:

1. Add key/value caching to decoder self-attention.
2. Add attention dropout and residual dropout tests.
3. Swap the ViT's sinusoidal `PositionalEncoding` back to a *learned* `nn.Parameter` pos_embed (the canonical ViT choice) and confirm the shapes still match.
4. Add Hungarian matching and detection losses to the DETR exercise.
5. Add skip connections or a feature-pyramid decoder to the segmentation exercise.
6. Replace the manual MHA with `torch.nn.functional.scaled_dot_product_attention` and compare outputs.
